# Notebook 2 — UCSF RMAC AML Augmentation Pipeline
**Dataset:** UCSF RMAC | **Format:** HDF5 | **Phase:** Arterial only | **Label:** AML only

---

## What This Notebook Does

This notebook performs image-level data augmentation on AML-labeled cases from the UCSF RMAC dataset. Only cases that contain the arterial phase are processed. Three augmented versions are created per eligible case — eroded mask, dilated mask, and flipped image+mask. All augmented files are saved as new HDF5 files with real voxel spacing and all original metadata preserved. Augmented files are treated as independent new images in Notebook 4 (UCSF feature extraction).

**Pipeline position:**
```
[Notebook 1] → [Notebook 2 — YOU ARE HERE] → [Notebook 3] → [Notebook 4]
 KiTS23 Aug     UCSF Augmentation             KiTS23 Extr   UCSF Extr
```

---

## Why Arterial Phase Only

Not all UCSF HDF5 files contain an arterial phase scan. Some cases only have `noncon`, `portven`, or `delay` phases. Since our study uses the arterial phase for radiomics extraction (matching the single phase available in KiTS23), cases without arterial phase cannot be augmented or used for feature extraction. This notebook detects and skips them automatically.

---

## Why Image-Level Augmentation Before Feature Extraction

AML is the minority class with severe imbalance (~9:1 RCC:AML). Image-level augmentation is performed before feature extraction so all features remain internally consistent. PyRadiomics computes each augmented sample's full feature set from its own image — biological relationships between features are preserved by construction. This is superior to SMOTE which interpolates already-extracted features independently and can create physically impossible combinations.

**Reference:** Bianchini et al. (2024), Journal of Digital Imaging
https://link.springer.com/article/10.1007/s10278-024-01013-0

---

## Augmentation Methods

| Method | Applied To | Factor | Rationale |
|--------|-----------|--------|-----------|
| Erosion | Mask only | 30% volume reduction | Simulates conservative segmentation |
| Dilation | Mask only | 30% volume increase | Simulates generous segmentation |
| Flip | Image + Mask | axis=0 | Mirror patient — valid due to kidney symmetry |

---

## Key Fixes vs. Original Pipeline

### Fix 1 — Arterial-Only Processing
Cases without `arterial` key or `arterial_pixdim` attribute are detected and skipped with a clear log message. Previously the pipeline either crashed or processed wrong phases silently.

### Fix 2 — Real Voxel Spacing Read and Preserved
Real spacing is stored in HDF5 file-level attributes as `arterial_pixdim` (e.g. `[0.761719, 0.761719, 5.0]`). The original pipeline never read this — it defaulted to `(1,1,1) mm`. This notebook reads real spacing and copies it into every augmented output file so Notebook 4 can set it correctly before PyRadiomics extraction.

### Fix 3 — Dataset Key Corrected
Augmented data is stored under key `'arterial'` (not `'image'` as in the original pipeline). Notebook 4 reads from `'arterial'` — using `'image'` caused silent extraction failures.

### Fix 4 — Size Mismatch Auto-Detection
Cases where image and mask array shapes do not match are detected automatically by shape comparison and skipped with a log entry.

---

## Confirmed HDF5 File Structure
```
case.hdf5  (cases WITH arterial phase)
  datasets:
    arterial   shape=(512, 512, N)  <- CT arterial phase  [USED]
    mask       shape=(512, 512, N)  <- tumor segmentation [USED]
    noncon     shape=(512, 512, N)  <- not used, copied
    portven    shape=(512, 512, N)  <- not used, copied
  attrs:
    arterial_pixdim  [0.761719, 0.761719, 5.0]  <- REAL SPACING
    mask_pixdim      [0.761719, 0.761719, 5.0]
    PID              'case_id_string'
    pathology        'angiomyolipoma'
    Manufacturer     'GE MEDICAL SYSTEMS Revolution HD'
    Patient Age      'XXX Y'
    Patient Sex      'F/M'

case.hdf5  (cases WITHOUT arterial phase — SKIPPED)
  datasets:
    delay    <- only other phases present
    noncon
    portven
    mask
  attrs:
    delay_pixdim, noncon_pixdim, portven_pixdim
    NO arterial_pixdim  <- detected and skipped
```

---

## Output Directory Structure
```
hdf5-Augmentation/
  eroded/case.hdf5   <- eroded mask, original arterial image, all attrs copied
  dilated/case.hdf5  <- dilated mask, original arterial image, all attrs copied
  flipped/case.hdf5  <- flipped image+mask, all attrs copied

All three folders will contain identical file lists.
Only arterial-phase cases appear in the output.
```

## 1. Install Dependencies

In [ ]:
!pip install h5py scipy numpy -q

## 2. Imports

In [ ]:
import os
import h5py
import numpy as np
from scipy import ndimage

print("Libraries loaded.")

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. Paths and Configuration

In [ ]:
SOURCE_DIR          = "/content/drive/MyDrive/hdf5/ucsf-rmac-dataset/hdf5_files/"
OUTPUT_DIR          = "/content/drive/MyDrive/hdf5-Augmentation"
PHASE               = "arterial"
PERTURBATION_FACTOR = 0.30

all_files = sorted([f for f in os.listdir(SOURCE_DIR) if f.endswith(".hdf5")])
print(f"Total HDF5 files found: {len(all_files)}")
print(f"First 5: {all_files[:5]}")

## 5. Pre-flight Check — Identify Arterial-Phase Cases

Scans all source files and separates arterial-phase cases from non-arterial cases before any augmentation runs. This gives a clear picture of exactly how many files will be processed.

In [ ]:
arterial_cases     = []
non_arterial_cases = []

for fname in all_files:
    fpath = os.path.join(SOURCE_DIR, fname)
    with h5py.File(fpath, "r") as f:
        has_arterial        = PHASE in f.keys()
        has_spacing         = f"{PHASE}_pixdim" in f.attrs
        has_mask            = "mask" in f.keys()

    if has_arterial and has_spacing and has_mask:
        arterial_cases.append(fname)
    else:
        non_arterial_cases.append((fname, has_arterial, has_spacing, has_mask))

print(f"Eligible for augmentation (arterial + spacing + mask): {len(arterial_cases)}")
print(f"Will be skipped                                       : {len(non_arterial_cases)}")
print()
if non_arterial_cases:
    print("Skipped cases:")
    print(f"  {'File':<30} {'arterial':>10} {'spacing':>10} {'mask':>6}")
    for fname, ha, hs, hm in non_arterial_cases:
        print(f"  {fname:<30} {str(ha):>10} {str(hs):>10} {str(hm):>6}")

## 6. Inspect One Arterial-Phase File — Confirm Structure

In [ ]:
if not arterial_cases:
    print("ERROR: No arterial-phase cases found. Check SOURCE_DIR and PHASE variable.")
else:
    sample = os.path.join(SOURCE_DIR, arterial_cases[0])
    with h5py.File(sample, "r") as f:
        print("File          :", arterial_cases[0])
        print("Dataset keys  :", list(f.keys()))
        print("Arterial shape:", f[PHASE].shape)
        print("Mask shape    :", f["mask"].shape)
        print("Attributes    :")
        for k, v in f.attrs.items():
            print(f"  {k}: {v}")
        spacing_key = f"{PHASE}_pixdim"
        print(f"\nSpacing '{spacing_key}': {f.attrs[spacing_key]}")

## 7. Core Augmentation Functions

### 7a. Erosion — mask only

In [ ]:
def get_3d_erosion(mask, factor=0.30):
    """
    Iteratively erodes the 3D mask until the target volume reduction is met.
    Applied to mask only. Arterial image array is passed through unchanged.
    Simulates a radiologist drawing a smaller, conservative tumor boundary.
    """
    original_area = np.sum(mask)
    target_area   = original_area * (1 - factor)
    perturbed     = np.copy(mask)
    struct        = ndimage.generate_binary_structure(3, 1)  # 6-connectivity

    while np.sum(perturbed) > target_area:
        prev_mask = np.copy(perturbed)
        perturbed = ndimage.binary_erosion(perturbed, structure=struct)
        if np.sum(perturbed) == 0:
            print("    Warning: erosion would eliminate mask. Stopping early.")
            return prev_mask
    return perturbed

### 7b. Dilation — mask only

In [ ]:
def get_3d_dilation(mask, factor=0.30):
    """
    Iteratively dilates the 3D mask until the target volume growth is met.
    Applied to mask only. Arterial image array is passed through unchanged.
    Simulates a radiologist drawing a larger, generous tumor boundary.
    """
    original_area = np.sum(mask)
    target_area   = original_area * (1 + factor)
    perturbed     = np.copy(mask)
    struct        = ndimage.generate_binary_structure(3, 1)

    while np.sum(perturbed) < target_area:
        perturbed = ndimage.binary_dilation(perturbed, structure=struct)
    return perturbed

### 7c. Flip — image and mask

In [ ]:
def run_flip_task(image_data, mask_data, axis=0):
    """
    Flips both image and mask along the given axis.
    HDF5 files have no affine matrix — spacing is stored in attrs
    and preserved by the save function.
    Both image and mask must be flipped together to preserve the
    spatial relationship between tumor texture and anatomy.
    """
    return (
        np.flip(image_data, axis=axis).copy(),
        np.flip(mask_data,  axis=axis).copy()
    )

## 8. Save Function — All Attributes Preserved

Stores augmented data under `'arterial'` key (not `'image'`). Copies all original file-level attributes including `arterial_pixdim` so Notebook 4 can read real spacing from augmented files.

In [ ]:
def save_hdf5_volume(augmented_image, augmented_mask, output_path,
                     original_attrs, original_file_path, phase="arterial"):
    """
    Saves an augmented image/mask pair as a new HDF5 file.

    - Stores data under the correct phase key ('arterial')
    - Copies ALL original file-level attributes including arterial_pixdim
    - Copies non-augmented phases (noncon, portven, delay) unchanged
    """
    os.makedirs(os.path.dirname(output_path), exist_ok=True)

    with h5py.File(output_path, "w") as f_out:

        # Write augmented arterial and mask under correct keys
        f_out.create_dataset(phase,  data=augmented_image, compression="gzip")
        f_out.create_dataset("mask", data=augmented_mask,  compression="gzip")

        # Copy all other phases unchanged
        with h5py.File(original_file_path, "r") as f_orig:
            for key in f_orig.keys():
                if key not in [phase, "mask"]:
                    f_out.create_dataset(key, data=f_orig[key][:], compression="gzip")

        # Copy ALL original attributes — including arterial_pixdim
        for key, val in original_attrs.items():
            f_out.attrs[key] = val

    print(f"  Saved: {output_path}")

## 9. Main Pipeline Controller

Processes only cases that passed the pre-flight check in Cell 5. Cases without arterial phase, missing spacing attribute, mismatched shapes, or empty masks are all skipped with clear log messages.

In [ ]:
def execute_augmentation_pipeline(source_dir, root_output_dir,
                                  phase="arterial", factor=0.30):
    """
    Iterates over all HDF5 files. For each file:
      - Skips if no arterial phase key
      - Skips if no arterial_pixdim spacing attribute
      - Skips if image/mask shape mismatch
      - Skips if mask is empty
      - Otherwise produces: eroded, dilated, flipped versions
    All three output folders will contain identical file lists.
    """
    os.makedirs(root_output_dir, exist_ok=True)
    cases   = sorted([f for f in os.listdir(source_dir) if f.endswith(".hdf5")])
    skipped = []
    success = []

    print(f"Starting augmentation for {len(cases)} source files...")
    print(f"Only arterial-phase cases with valid spacing will be processed.\n")

    for filename in cases:
        file_path = os.path.join(source_dir, filename)
        print(f"Processing: {filename}")

        try:
            with h5py.File(file_path, "r") as f:

                # --- Check 1: arterial phase exists ---
                if phase not in f.keys():
                    print(f"  SKIPPED: no '{phase}' dataset — phases available: {list(f.keys())}")
                    skipped.append((filename, f"no {phase} phase"))
                    continue

                # --- Check 2: mask exists ---
                if "mask" not in f.keys():
                    print(f"  SKIPPED: no 'mask' dataset.")
                    skipped.append((filename, "no mask"))
                    continue

                # --- Check 3: spacing attribute exists ---
                spacing_key = f"{phase}_pixdim"
                if spacing_key not in f.attrs:
                    print(f"  SKIPPED: '{spacing_key}' attribute not found — "
                          f"no arterial spacing available.")
                    skipped.append((filename, f"missing {spacing_key}"))
                    continue

                image_array = f[phase][:]
                mask_array  = f["mask"][:]
                attrs       = dict(f.attrs)  # includes arterial_pixdim

            # --- Check 4: shape mismatch ---
            if image_array.shape != mask_array.shape:
                print(f"  SKIPPED: image {image_array.shape} != mask {mask_array.shape}")
                skipped.append((filename, "shape mismatch"))
                continue

            # --- Check 5: empty mask ---
            if np.sum(mask_array) == 0:
                print(f"  SKIPPED: mask is empty.")
                skipped.append((filename, "empty mask"))
                continue

            # --- Task A: Erosion ---
            print("  [A] Erosion...")
            eroded = get_3d_erosion(mask_array, factor)
            save_hdf5_volume(
                image_array, eroded,
                os.path.join(root_output_dir, "eroded", filename),
                attrs, file_path, phase
            )

            # --- Task B: Dilation ---
            print("  [B] Dilation...")
            dilated = get_3d_dilation(mask_array, factor)
            save_hdf5_volume(
                image_array, dilated,
                os.path.join(root_output_dir, "dilated", filename),
                attrs, file_path, phase
            )

            # --- Task C: Flip ---
            print("  [C] Flip...")
            f_img, f_msk = run_flip_task(image_array, mask_array, axis=0)
            save_hdf5_volume(
                f_img, f_msk,
                os.path.join(root_output_dir, "flipped", filename),
                attrs, file_path, phase
            )

            success.append(filename)
            print(f"  Done: {filename}\n")

        except Exception as e:
            print(f"  ERROR: {e}")
            skipped.append((filename, str(e)))

    # --- Final summary ---
    print("=" * 55)
    print(f"Augmentation complete.")
    print(f"  Successfully processed : {len(success)}")
    print(f"  Skipped                : {len(skipped)}")
    if skipped:
        print(f"  Skip details:")
        for s, r in skipped:
            print(f"    {s}: {r}")

    return success, skipped

## 10. Run the Pipeline

In [ ]:
success, skipped = execute_augmentation_pipeline(
    source_dir      = SOURCE_DIR,
    root_output_dir = OUTPUT_DIR,
    phase           = PHASE,
    factor          = PERTURBATION_FACTOR
)

## 11. Verify Output — File Counts and Spacing Preserved

All three folders must contain identical file counts and lists. Spacing in augmented files must match the original source file spacing.

In [ ]:
spacing_key = f"{PHASE}_pixdim"

# --- File count check ---
print("=== File Count per Augmentation Type ===")
folder_files = {}
for aug in ["eroded", "dilated", "flipped"]:
    aug_dir = os.path.join(OUTPUT_DIR, aug)
    if os.path.exists(aug_dir):
        files = sorted([f for f in os.listdir(aug_dir) if f.endswith(".hdf5")])
        folder_files[aug] = files
        print(f"  {aug:8s}: {len(files)} files")
    else:
        folder_files[aug] = []
        print(f"  {aug:8s}: DIRECTORY NOT FOUND")

# Confirm all three folders have identical file lists
sets = [set(v) for v in folder_files.values()]
all_match = len(sets) > 0 and sets[0] == sets[1] == sets[2]
print(f"\nAll three folders identical: {'YES' if all_match else 'NO — check skipped cases above'}")

# --- Spacing preservation check ---
print("\n=== Spacing Preservation Check (first processed case) ===")
if success:
    sample_fname = success[0]

    with h5py.File(os.path.join(SOURCE_DIR, sample_fname), "r") as f:
        orig_spacing = f.attrs.get(spacing_key, "NOT FOUND")
    print(f"Original spacing : {orig_spacing}")

    for aug in ["eroded", "dilated", "flipped"]:
        aug_path = os.path.join(OUTPUT_DIR, aug, sample_fname)
        if os.path.exists(aug_path):
            with h5py.File(aug_path, "r") as f:
                aug_spacing = f.attrs.get(spacing_key, "NOT FOUND")
                has_phase   = PHASE in f.keys()
                has_mask    = "mask" in f.keys()
            match = np.allclose(orig_spacing, aug_spacing)
            print(f"  {aug:8s}: spacing={'MATCH' if match else 'MISMATCH'} {aug_spacing} | "
                  f"'{PHASE}'={has_phase} | 'mask'={has_mask}")
        else:
            print(f"  {aug:8s}: FILE NOT FOUND")
else:
    print("No successfully processed cases to verify.")